In [1]:
import pandas as pd

# Upload your files 
files = [
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Original_Dataset.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Symptom_Weights.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Disease_Description.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\medicine.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Specialist.csv',
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv'
]

for file in files:
    try:
        df = pd.read_csv(file)
        print(f"\n{'='*60}")
        print(f" File: {file}")
        print(f"{'='*60}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"\nFirst 3 rows:")
        print(df.head(3))
    except Exception as e:
        print(f"Error reading {file}: {e}")


 File: C:\Users\Riyad\projects\medical-assistand-ml\Original_Dataset.csv
Shape: (4920, 18)
Columns: ['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']

First 3 rows:
            Disease   Symptom_1              Symptom_2              Symptom_3  \
0  Fungal infection     itching              skin_rash   nodal_skin_eruptions   
1  Fungal infection   skin_rash   nodal_skin_eruptions    dischromic _patches   
2  Fungal infection     itching   nodal_skin_eruptions    dischromic _patches   

              Symptom_4 Symptom_5 Symptom_6 Symptom_7 Symptom_8 Symptom_9  \
0   dischromic _patches       NaN       NaN       NaN       NaN       NaN   
1                   NaN       NaN       NaN       NaN       NaN       NaN   
2                   NaN       NaN       NaN       NaN       NaN       NaN   

  Symptom

In [2]:
!pip install scikit-learn pandas numpy joblib -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully!\n")

 Libraries imported successfully!



In [4]:
BASE = r'C:\Users\Riyad\projects\medical-assistand-ml'

df_original = pd.read_csv(f'{BASE}\\Original_Dataset.csv')
df_symptom_weights = pd.read_csv(f'{BASE}\\Symptom_Weights.csv')
df_disease_desc = pd.read_csv(f'{BASE}\\Disease_Description.csv')
df_medicine = pd.read_csv(f'{BASE}\\medicine.csv')
df_doctor_spec = pd.read_csv(f'{BASE}\\Doctor_Specialist.csv')
df_doctor_vs_disease = pd.read_csv(
    f'{BASE}\\Doctor_Versus_Disease.csv',
    encoding='windows-1252',
    header=None,
    names=['Disease', 'Doctor_Specialist']
)

print(f" Original Dataset:    {df_original.shape}")
print(f" Symptom Weights:     {df_symptom_weights.shape}")
print(f" Disease Description: {df_disease_desc.shape}")
print(f" Medicine Dataset:    {df_medicine.shape}")
print(f" Doctor Specialist:   {df_doctor_spec.shape}")
print(f" Doctor Vs Disease:   {df_doctor_vs_disease.shape}")
print(f"\n{df_doctor_vs_disease.head(10)}")

 Original Dataset:    (4920, 18)
 Symptom Weights:     (130, 2)
 Disease Description: (41, 2)
 Medicine Dataset:    (21714, 10)
 Doctor Specialist:   (19, 1)
 Doctor Vs Disease:   (41, 2)

            Disease Doctor_Specialist
0     Drug Reaction         Allergist
1           Allergy         Allergist
2     Hypertension       Cardiologist
3      Heart attack      Cardiologist
4         Psoriasis     Dermatologist
5       Chicken pox     Dermatologist
6              Acne     Dermatologist
7          Impetigo     Dermatologist
8  Fungal infection     Dermatologist
9    Hypothyroidism   Endocrinologist


In [5]:
print(" Preprocessing data...")

# Get all unique symptoms from the dataset
symptom_columns = [col for col in df_original.columns if col.startswith('Symptom_')]
print(f" Found {len(symptom_columns)} symptom columns")

# Get all unique symptoms (excluding NaN)
all_symptoms = set()
for col in symptom_columns:
    symptoms = df_original[col].dropna().unique()
    all_symptoms.update(symptoms)

all_symptoms = sorted(list(all_symptoms))
print(f" Total unique symptoms: {len(all_symptoms)}\n")

# Create symptom to index mapping
symptom_to_index = {symptom: idx for idx, symptom in enumerate(all_symptoms)}

 Preprocessing data...
 Found 17 symptom columns
 Total unique symptoms: 131



In [6]:
print(" Creating feature matrix...")

def create_feature_vector(row, symptom_columns, symptom_to_index):
    """Convert symptoms to binary feature vector"""
    feature_vector = np.zeros(len(symptom_to_index))

    for col in symptom_columns:
        symptom = row[col]
        if pd.notna(symptom) and symptom in symptom_to_index:
            feature_vector[symptom_to_index[symptom]] = 1

    return feature_vector

# Create features (X) and labels (y)
X = []
y = []

for idx, row in df_original.iterrows():
    feature_vec = create_feature_vector(row, symptom_columns, symptom_to_index)
    X.append(feature_vec)
    y.append(row['Disease'])

    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1}/{len(df_original)} rows...")

X = np.array(X)
y = np.array(y)

print(f"\n Feature matrix shape: {X.shape}")
print(f" Labels shape: {y.shape}")
print(f" Number of unique diseases: {len(np.unique(y))}\n")

 Creating feature matrix...
Processed 1000/4920 rows...
Processed 2000/4920 rows...
Processed 3000/4920 rows...
Processed 4000/4920 rows...

 Feature matrix shape: (4920, 131)
 Labels shape: (4920,)
 Number of unique diseases: 41



In [7]:
print(" Encoding disease labels...")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f" Encoded labels shape: {y_encoded.shape}\n")

 Encoding disease labels...
 Encoded labels shape: (4920,)



In [8]:
print(" Splitting dataset into train and test...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f" Training set: {X_train.shape}")
print(f" Test set: {X_test.shape}\n")

 Splitting dataset into train and test...
 Training set: (3936, 131)
 Test set: (984, 131)



In [9]:
print(" Training Random Forest model...")


model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

model.fit(X_train, y_train)

print("\n Model training completed!\n")

 Training Random Forest model...

 Model training completed!



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.0s finished


In [10]:
print("Evaluating model...")

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Model Accuracy: {accuracy * 100:.2f}%\n")

print(" Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Evaluating model...

 Model Accuracy: 100.00%

 Classification Report:

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        24
                                   AIDS       1.00      1.00      1.00        24
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        24
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        24
                       Bronchial Asthma       1.00      1.00      1.00        24
                   Cervical spondylosis       1.00      1.00      1.00        24
                            Chicken pox       1.00      1.00      1.00        24
                    Chronic cholestasis       1.00      1.00      1.00        24
                            Common C

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [11]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

In [12]:
joblib.dump(model,            f'{BASE}\\disease_prediction_model.pkl')
joblib.dump(label_encoder,    f'{BASE}\\label_encoder.pkl')
joblib.dump(symptom_to_index, f'{BASE}\\symptom_to_index.pkl')
joblib.dump(all_symptoms,     f'{BASE}\\all_symptoms.pkl')
joblib.dump(df_disease_desc,  f'{BASE}\\disease_descriptions.pkl')
joblib.dump(df_symptom_weights, f'{BASE}\\symptom_weights.pkl')

['C:\\Users\\Riyad\\projects\\medical-assistand-ml\\symptom_weights.pkl']

In [13]:
doctor_map = dict(zip(df_doctor_vs_disease['Disease'], df_doctor_vs_disease['Doctor_Specialist']))
joblib.dump(doctor_map,         f'{BASE}\\doctor_map.pkl')

['C:\\Users\\Riyad\\projects\\medical-assistand-ml\\doctor_map.pkl']

In [14]:
print("\n Testing prediction with sample symptoms...")

def predict_disease(symptoms_list, model, label_encoder, symptom_to_index):
    """Predict disease from list of symptoms"""
    feature_vector = np.zeros(len(symptom_to_index))

    for symptom in symptoms_list:
        if symptom in symptom_to_index:
            feature_vector[symptom_to_index[symptom]] = 1

    feature_vector = feature_vector.reshape(1, -1)
    prediction = model.predict(feature_vector)
    disease = label_encoder.inverse_transform(prediction)[0]

    # Get probability
    probabilities = model.predict_proba(feature_vector)[0]
    confidence = max(probabilities) * 100

    return disease, confidence

# Test with sample symptoms
test_symptoms = ['itching', 'skin_rash', 'nodal_skin_eruptions']
predicted_disease, confidence = predict_disease(
    test_symptoms, model, label_encoder, symptom_to_index
)

print(f"\n Input Symptoms: {test_symptoms}")
print(f" Predicted Disease: {predicted_disease}")
print(f" Confidence: {confidence:.2f}%")

# Get description
if predicted_disease in df_disease_desc['Disease'].values:
    desc = df_disease_desc[df_disease_desc['Disease'] == predicted_disease]['Description'].values[0]
    print(f"Description: {desc[:500]}...")

print("\n" + "="*60)
print("MODEL TRAINING COMPLETE!")
print("="*60)
print("\n Download these files for API:")
print("   - disease_prediction_model.pkl")
print("   - label_encoder.pkl")
print("   - symptom_to_index.pkl")
print("   - all_symptoms.pkl")
print("   - disease_descriptions.pkl")
print("   - medicine_data.pkl")
print("\n Next step: Create Flask/FastAPI for deployment!")


 Testing prediction with sample symptoms...

 Input Symptoms: ['itching', 'skin_rash', 'nodal_skin_eruptions']
 Predicted Disease: Drug Reaction
 Confidence: 7.28%
Description: An adverse drug reaction (ADR) is an injury caused by taking medication. ADRs may occur following a single dose or prolonged administration of a drug or result from the combination of two or more drugs....

MODEL TRAINING COMPLETE!

 Download these files for API:
   - disease_prediction_model.pkl
   - label_encoder.pkl
   - symptom_to_index.pkl
   - all_symptoms.pkl
   - disease_descriptions.pkl
   - medicine_data.pkl

 Next step: Create Flask/FastAPI for deployment!


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [15]:
!pip install fastapi uvicorn pyngrok python-multipart -q
!pip install rapidfuzz nltk scikit-learn joblib -q
!pip install nest-asyncio -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/627.2 kB ? eta -:--:--
     - ------------------------------------- 30.7/627.2 kB 1.3 MB/s eta 0:00:01
     ------ ------------------------------- 112.6/627.2 kB 1.3 MB/s eta 0:00:01
     ---------- --------------------------- 174.1/627.2 kB 1.2 MB/s eta 0:00:01
     ------------- ------------------------ 225.3/627.2 kB 1.1 MB/s eta 0:00:01
     ----------------- -------------------- 286.7/627.2 kB 1.4 MB/s eta 0:00:01
     ------------------------- ------------ 419.8/627.2 kB 1.5 MB/s eta 0:00:01
     ------------------------------- ------ 522.2/627.2 kB 1.6 MB/s eta 0:00:01
     -------------------------------------  624.6/627.2 kB 1.6 MB/s eta 0:00:01
     -------------------------------------- 627.2/627.2 kB 1.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
!pip install chardet


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import chardet
with open('C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv', 'rb') as f:
    result = chardet.detect(f.read())
    print(result['encoding'])

WINDOWS-1252


In [19]:
import pandas as pd

df_doctor_vs_disease = pd.read_csv(
    'C:\\Users\\Riyad\\projects\\medical-assistand-ml\\Doctor_Versus_Disease.csv',
    encoding='windows-1252'
)

print(df_doctor_vs_disease.shape)
print(df_doctor_vs_disease.columns.tolist())
print(df_doctor_vs_disease.head(10))

(40, 2)
['Drug Reaction', 'Allergist']
      Drug Reaction        Allergist
0           Allergy        Allergist
1     Hypertension      Cardiologist
2      Heart attack     Cardiologist
3         Psoriasis    Dermatologist
4       Chicken pox    Dermatologist
5              Acne    Dermatologist
6          Impetigo    Dermatologist
7  Fungal infection    Dermatologist
8    Hypothyroidism  Endocrinologist
9         Diabetes   Endocrinologist


In [20]:

print(" Importing libraries...")

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
import joblib
import numpy as np
from rapidfuzz import fuzz, process
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
import logging
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import nltk
nltk.download('punkt_tab')

# Apply nest_asyncio to run uvicorn in Colab
nest_asyncio.apply()

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt'),
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
    nltk.download('stopwords')

print(" Libraries imported!\n")

 Importing libraries...
 Libraries imported!



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Riyad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [21]:
!pip install rapidfuzz -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [23]:
try:
    model = joblib.load(f'{BASE}\\disease_prediction_model.pkl')
    label_encoder = joblib.load(f'{BASE}\\label_encoder.pkl')
    symptom_to_index = joblib.load(f'{BASE}\\symptom_to_index.pkl')
    all_symptoms = joblib.load( f'{BASE}\\all_symptoms.pkl')
    disease_descriptions = joblib.load(f'{BASE}\\disease_descriptions.pkl')
    medicine_data = joblib.load( f'{BASE}\\medicine_data.pkl')
    doctor_map = joblib.load(f'{BASE}\\doctor_map.pkl')

    print(" All models loaded successfully!\n")
except Exception as e:
    print(f" Error loading models: {e}")
    print("\nMake sure these files exist in your Google Drive:")
    print("  - disease_prediction_model.pkl")
    print("  - label_encoder.pkl")
    print("  - symptom_to_index.pkl")
    print("  - all_symptoms.pkl")
    print("  - disease_descriptions.pkl")
    print("  - medicine_data.pkl")
    print("  - doctor_map.pkl")
    raise

 All models loaded successfully!



In [24]:
print(" Initializing FastAPI...\n")

app = FastAPI(
    title="Medical Assistant API",
    description="AI-powered medical symptom checker",
    version="2.0.0"
)

# Enable CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

 Initializing FastAPI...



In [25]:
class SymptomInput(BaseModel):
    symptoms: str

class SymptomListInput(BaseModel):
    symptoms: List[str]

class DiseaseResponse(BaseModel):
    success: bool
    disease: str
    confidence: float
    description: str
    matched_symptoms: List[str]
    input_symptoms: List[str]
    suggested_medicines: List[str]
    precautions: List[str]
    doctor_specialty: Optional[str]
    disclaimer: str

class HealthResponse(BaseModel):
    status: str
    message: str
    total_symptoms: int
    total_diseases: int

In [26]:
import os
import re
import json
import logging
import numpy as np
import joblib
import nltk
import uvicorn

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any
from rapidfuzz import fuzz, process
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from groq import Groq
from pyngrok import ngrok

# NLTK download
for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.data.find(f'tokenizers/{pkg}')
    except LookupError:
        nltk.download(pkg)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Riyad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# API Keys (replace with your actual keys)

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
NGROK_TOKEN  = os.getenv("NGROK_TOKEN")
BASE = r'C:\Users\Riyad\projects\medical-assistand-ml'

groq_client = Groq(api_key=GROQ_API_KEY)

# Load pkl files
try:
    model                = joblib.load(f'{BASE}\\disease_prediction_model.pkl')
    label_encoder        = joblib.load(f'{BASE}\\label_encoder.pkl')
    symptom_to_index     = joblib.load(f'{BASE}\\symptom_to_index.pkl')
    all_symptoms         = joblib.load(f'{BASE}\\all_symptoms.pkl')
    disease_descriptions = joblib.load(f'{BASE}\\disease_descriptions.pkl')
    doctor_map           = joblib.load(f'{BASE}\\doctor_map.pkl')
    print("All models loaded!")
except Exception as e:
    print(f" Error: {e}")
    raise

All models loaded!


In [29]:
class SymptomInput(BaseModel):
    symptoms: str

class SymptomListInput(BaseModel):
    symptoms: List[str]

class DiseaseResponse(BaseModel):
    success: bool
    disease: str
    confidence: float
    description: str
    matched_symptoms: List[str]
    input_symptoms: List[str]
    suggested_medicines: List[str]
    precautions: List[str]
    doctor_specialty: str
    source: str
    disclaimer: str

class HealthResponse(BaseModel):
    status: str
    message: str
    total_symptoms: int
    total_diseases: int

In [29]:
def ask_groq_ai(user_symptoms: str) -> Dict[str, Any]:
    prompt = f"""You are a medical assistant AI. A patient describes these symptoms: "{user_symptoms}"

Respond ONLY in this JSON format (no extra text):
{{
    "disease": "Most likely disease name",
    "confidence": 70,
    "description": "Brief description (2-3 sentences)",
    "matched_symptoms": ["symptom1", "symptom2"],
    "suggested_medicines": ["Medicine 1", "Medicine 2", "Medicine 3"],
    "precautions": ["Precaution 1", "Precaution 2", "Precaution 3"],
    "doctor_specialty": "Relevant specialist"
}}
Rules:
- Be medically accurate but conservative
- Confidence between 50-85
- Generic medicine names only"""

    try:
        response = groq_client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=800
        )
        raw = response.choices[0].message.content.strip()
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            result['source'] = 'groq_ai'
            return result
        raise ValueError("No JSON in response")
    except Exception as e:
        logger.error(f"Groq error: {e}")
        return {
            "disease": "Unknown — Please see a doctor",
            "confidence": 0,
            "description": "Could not identify disease. Please consult a doctor.",
            "matched_symptoms": [],
            "suggested_medicines": ["Consult a doctor for proper medication"],
            "precautions": ["See a doctor immediately", "Do not self-medicate",
                           "Rest and stay hydrated", "Monitor symptoms"],
            "doctor_specialty": "General Physician",
            "source": "fallback"
        }

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^\w\s_]', ' ', text)
    return ' '.join(text.split())

def extract_symptoms_from_text(text: str, all_symptoms: List[str]) -> List[str]:
    text_clean = clean_text(text)
    tokens = word_tokenize(text_clean)
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    matched = set()

    for symptom in all_symptoms:
        s_clean = clean_text(symptom)
        if s_clean in text_clean:
            matched.add(symptom)
            continue
        if fuzz.partial_ratio(s_clean, text_clean) > 80:
            matched.add(symptom)
            continue
        s_tokens = s_clean.replace('_', ' ').split()
        if len(s_tokens) > 1 and all(t in text_clean for t in s_tokens):
            matched.add(symptom)

    return list(matched)

def fuzzy_match_symptoms(input_symptoms: List[str], all_symptoms: List[str]) -> List[str]:
    matched = []
    all_clean = [clean_text(s) for s in all_symptoms]
    for inp in input_symptoms:
        inp_clean = clean_text(inp)
        best = process.extractOne(inp_clean, all_clean, scorer=fuzz.ratio)
        if best and best[1] > 70:
            idx = all_clean.index(best[0])
            matched.append(all_symptoms[idx])
            logger.info(f"Matched '{inp}' → '{all_symptoms[idx]}' ({best[1]}%)")
    return matched

def predict_disease(symptoms: List[str]) -> Dict[str, Any]:
    if not symptoms:
        raise ValueError("No symptoms found")
    feature_vector = np.zeros(len(symptom_to_index))
    for s in symptoms:
        if s in symptom_to_index:
            feature_vector[symptom_to_index[s]] = 1
    fv = feature_vector.reshape(1, -1)
    prediction = model.predict(fv)
    proba = model.predict_proba(fv)[0]
    disease = label_encoder.inverse_transform(prediction)[0]
    confidence = max(proba) * 100
    return {"disease": disease, "confidence": confidence}

def get_disease_description(disease: str) -> str:
    row = disease_descriptions[disease_descriptions['Disease'] == disease]
    return row['Description'].values[0] if not row.empty else "No description available."

# ── Upgraded medicine map ──
DISEASE_MEDICINE_MAP = {
    "Fungal infection":        ["Fluconazole", "Clotrimazole cream", "Terbinafine"],
    "Allergy":                 ["Cetirizine", "Loratadine", "Fexofenadine"],
    "GERD":                    ["Omeprazole", "Pantoprazole", "Antacids"],
    "Chronic cholestasis":     ["Ursodeoxycholic acid", "Cholestyramine"],
    "Drug Reaction":           ["Antihistamines", "Corticosteroids"],
    "Peptic ulcer diseae":     ["Omeprazole", "Amoxicillin", "Antacids"],
    "AIDS":                    ["Antiretroviral therapy (ART)"],
    "Diabetes":                ["Metformin", "Insulin", "Glibenclamide"],
    "Gastroenteritis":         ["ORS", "Zinc supplements", "Loperamide"],
    "Bronchial Asthma":        ["Salbutamol inhaler", "Budesonide inhaler"],
    "Hypertension":            ["Amlodipine", "Enalapril", "Losartan"],
    "Migraine":                ["Sumatriptan", "Ibuprofen", "Propranolol"],
    "Cervical spondylosis":    ["Ibuprofen", "Diclofenac", "Muscle relaxants"],
    "Paralysis (brain hemorrhage)": ["Aspirin", "Atorvastatin"],
    "Jaundice":                ["Liver tonics", "Adequate hydration"],
    "Malaria":                 ["Chloroquine", "Artemether-Lumefantrine"],
    "Chicken pox":             ["Acyclovir", "Calamine lotion", "Paracetamol"],
    "Dengue":                  ["Paracetamol", "ORS", "Avoid NSAIDs"],
    "Typhoid":                 ["Ciprofloxacin", "Azithromycin"],
    "Hepatitis A":             ["Rest", "Adequate nutrition"],
    "Hepatitis B":             ["Tenofovir", "Entecavir"],
    "Hepatitis C":             ["Sofosbuvir", "Ribavirin"],
    "Hepatitis D":             ["Interferon alfa"],
    "Hepatitis E":             ["Supportive care", "Rest"],
    "Alcoholic hepatitis":     ["Stop alcohol", "Corticosteroids"],
    "Tuberculosis":            ["Isoniazid", "Rifampicin", "Pyrazinamide"],
    "Common Cold":             ["Paracetamol", "Cetirizine", "Vitamin C"],
    "Pneumonia":               ["Amoxicillin", "Azithromycin", "Ceftriaxone"],
    "Dimorphic hemmorhoids(piles)": ["Sitz bath", "Hydrocortisone cream"],
    "Heart attack":            ["Aspirin", "Clopidogrel", "Atorvastatin"],
    "Varicose veins":          ["Compression stockings", "Sclerotherapy"],
    "Hypothyroidism":          ["Levothyroxine"],
    "Hyperthyroidism":         ["Methimazole", "Beta-blockers"],
    "Hypoglycemia":            ["Glucose tablets", "Fruit juice"],
    "Osteoarthritis":          ["Paracetamol", "Ibuprofen", "Glucosamine"],
    "Arthritis":               ["Methotrexate", "NSAIDs", "Hydroxychloroquine"],
    "(vertigo) Paroymsal Positional Vertigo": ["Betahistine", "Meclizine"],
    "Acne":                    ["Benzoyl peroxide", "Clindamycin gel", "Retinoids"],
    "Urinary tract infection": ["Nitrofurantoin", "Ciprofloxacin", "Trimethoprim"],
    "Psoriasis":               ["Methotrexate", "Calcipotriol cream", "Coal tar"],
    "Impetigo":                ["Mupirocin cream", "Amoxicillin"],
}

def get_medicine_suggestions(disease: str) -> List[str]:
    medicines = DISEASE_MEDICINE_MAP.get(disease, [])
    if not medicines:
        for key in DISEASE_MEDICINE_MAP:
            if key.lower() in disease.lower() or disease.lower() in key.lower():
                return DISEASE_MEDICINE_MAP[key]
        return ["Consult a doctor for proper prescription"]
    return medicines

def get_doctor_specialty(disease: str) -> str:
    
    if disease in doctor_map:
        return doctor_map[disease]
   
    keyword_map = {
        'skin': 'Dermatologist', 'fungal': 'Dermatologist',
        'heart': 'Cardiologist', 'cardiac': 'Cardiologist',
        'diabetes': 'Endocrinologist', 'thyroid': 'Endocrinologist',
        'liver': 'Hepatologist', 'hepatitis': 'Hepatologist',
        'kidney': 'Nephrologist', 'urin': 'Urologist',
        'lung': 'Pulmonologist', 'asthma': 'Pulmonologist',
        'stomach': 'Gastroenterologist', 'gastro': 'Gastroenterologist',
        'bone': 'Orthopedic', 'joint': 'Orthopedic',
        'brain': 'Neurologist', 'migraine': 'Neurologist',
        'mental': 'Psychiatrist', 'anxiety': 'Psychiatrist',
        'eye': 'Ophthalmologist', 'ear': 'ENT', 'throat': 'ENT',
    }
    for kw, spec in keyword_map.items():
        if kw in disease.lower():
            return spec
    return "General Physician"

def get_precautions(disease: str) -> List[str]:
    return [
        "Consult a qualified medical doctor immediately",
        "Do not self-medicate without professional advice",
        "Maintain proper hygiene and rest",
        "Stay hydrated and eat nutritious food",
        "Monitor your symptoms and seek help if they worsen"
    ]

In [31]:
app = FastAPI(
    title="Medical Assistant API",
    description="AI-powered symptom checker with Groq AI fallback",
    version="3.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/", response_model=HealthResponse)
async def root():
    return {
        "status": "healthy",
        "message": "Medical Assistant API v3.0 (Groq AI fallback enabled)",
        "total_symptoms": len(all_symptoms),
        "total_diseases": len(label_encoder.classes_)
    }

@app.get("/health", response_model=HealthResponse)
async def health_check():
    return {
        "status": "healthy",
        "message": "All systems operational",
        "total_symptoms": len(all_symptoms),
        "total_diseases": len(label_encoder.classes_)
    }

@app.get("/symptoms", response_model=List[str])
async def get_all_symptoms():
    return all_symptoms

@app.post("/predict", response_model=DiseaseResponse)
async def predict_from_text(input_data: SymptomInput):
    try:
        logger.info(f"Input: {input_data.symptoms}")
        extracted = extract_symptoms_from_text(input_data.symptoms, all_symptoms)
        logger.info(f"Extracted: {extracted}")

        # groq_ai fallback if no symptoms extracted
        if not extracted:
            logger.info("→ Groq AI fallback")
            groq_result = ask_groq_ai(input_data.symptoms)
            return {
                "success": True,
                "disease": groq_result["disease"],
                "confidence": groq_result["confidence"],
                "description": groq_result["description"],
                "matched_symptoms": groq_result["matched_symptoms"],
                "input_symptoms": [input_data.symptoms],
                "suggested_medicines": groq_result["suggested_medicines"],
                "precautions": groq_result["precautions"],
                "doctor_specialty": groq_result["doctor_specialty"],
                "source": groq_result["source"],
                "disclaimer": " Groq AI prediction. NOT a medical diagnosis. Consult a doctor."
            }

        prediction = predict_disease(extracted)
        disease = prediction['disease']
        return {
            "success": True,
            "disease": disease,
            "confidence": round(prediction['confidence'], 2),
            "description": get_disease_description(disease),
            "matched_symptoms": extracted,
            "input_symptoms": [input_data.symptoms],
            "suggested_medicines": get_medicine_suggestions(disease),
            "precautions": get_precautions(disease),
            "doctor_specialty": get_doctor_specialty(disease),
            "source": "dataset",
            "disclaimer": " AI prediction. NOT a medical diagnosis. Consult a doctor."
        }
    except Exception as e:
        logger.error(f"Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict-from-list", response_model=DiseaseResponse)
async def predict_from_list(input_data: SymptomListInput):
    try:
        matched = fuzzy_match_symptoms(input_data.symptoms, all_symptoms)

        if not matched:
            groq_result = ask_groq_ai(", ".join(input_data.symptoms))
            return {
                "success": True,
                "disease": groq_result["disease"],
                "confidence": groq_result["confidence"],
                "description": groq_result["description"],
                "matched_symptoms": groq_result["matched_symptoms"],
                "input_symptoms": input_data.symptoms,
                "suggested_medicines": groq_result["suggested_medicines"],
                "precautions": groq_result["precautions"],
                "doctor_specialty": groq_result["doctor_specialty"],
                "source": groq_result["source"],
                "disclaimer": " Groq AI prediction. NOT a medical diagnosis. Consult a doctor."
            }

        prediction = predict_disease(matched)
        disease = prediction['disease']
        return {
            "success": True,
            "disease": disease,
            "confidence": round(prediction['confidence'], 2),
            "description": get_disease_description(disease),
            "matched_symptoms": matched,
            "input_symptoms": input_data.symptoms,
            "suggested_medicines": get_medicine_suggestions(disease),
            "precautions": get_precautions(disease),
            "doctor_specialty": get_doctor_specialty(disease),
            "source": "dataset",
            "disclaimer": " AI prediction. NOT a medical diagnosis. Consult a doctor."
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [32]:
if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"\n Public URL: {public_url}")
    print(f" Docs: {public_url}/docs\n")
    


2026-03-11 02:51:16,953 - INFO - Updating authtoken for default "config_path" of "ngrok_path": C:\Users\Riyad\AppData\Local\ngrok\ngrok.exe
2026-03-11 02:51:20,069 - INFO - Opening tunnel named: http-8000-4a9d8ee7-a64a-44d1-be27-82ed4eb0b491
2026-03-11 02:51:20,107 - INFO - t=2026-03-11T02:51:20+0600 lvl=info msg="no configuration paths supplied"
2026-03-11 02:51:20,108 - INFO - t=2026-03-11T02:51:20+0600 lvl=info msg="using configuration at default config path" path=C:\\Users\\Riyad\\AppData\\Local/ngrok/ngrok.yml
2026-03-11 02:51:20,108 - INFO - t=2026-03-11T02:51:20+0600 lvl=info msg="open config file" path=C:\\Users\\Riyad\\AppData\\Local\\ngrok\\ngrok.yml err=nil
2026-03-11 02:51:20,109 - INFO - t=2026-03-11T02:51:20+0600 lvl=info msg="FIPS 140 mode" enabled=false
2026-03-11 02:51:20,124 - INFO - t=2026-03-11T02:51:20+0600 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-03-11 02:51:20,703 - INFO - t=2026-03-11T02:51:20+0600 lvl=info msg="update 


 Public URL: NgrokTunnel: "https://predeterminate-falsely-annabel.ngrok-free.dev" -> "http://localhost:8000"
 Docs: NgrokTunnel: "https://predeterminate-falsely-annabel.ngrok-free.dev" -> "http://localhost:8000"/docs



2026-03-11 02:51:21,739 - INFO - t=2026-03-11T02:51:21+0600 lvl=info msg=end pg=/api/tunnels id=5b2e6bfff874e3dc status=201 dur=526.246ms


In [ ]:
import threading

def start_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Run uvicorn in a separate thread
thr = threading.Thread(target=start_uvicorn)
thr.start()

INFO:     Started server process [18396]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
